In [1]:
import requests
from config import (
    POP_FLOW_BASE_URL,
    POP_FLOW_SERVICE_KEY,
    POP_FLOW_COLUMNS,
    START_DATE,
    END_DATE,
    BATCH_MONTHS,
    DB_URL)
import pandas as pd
import xml.etree.ElementTree as ET
from sqlalchemy import create_engine, text
import time
from requests.exceptions import ReadTimeout

In [7]:
def create_db_engine():
    return create_engine(
        DB_URL,
        pool_pre_ping=True)

def read_districts_csv():
    df = pd.read_csv('/workspaces/korea-real-estate-population-movement/data/seoul_district_codes.csv')

    required = {"Districts","Code"}

    if not required.issubset(df.columns):
        raise ValueError("CSV missing required columns.")

    df["Districts"] = df["Districts"].str.strip()
    df["Code"] = pd.to_numeric(df["Code"], errors="raise")

    return dict(zip(df['Districts'],df['Code']))

def return_df(root):
    items = root.findall(".//item")
    data = []

    for item in items:
        row = {}

        for child in item:
            if child.tag in POP_FLOW_COLUMNS:
                row[child.tag] = child.text

        data.append(row)

    return pd.DataFrame(data)

def rename_columns(df):
    return df.rename(columns={
    "statsYm":"date",
    "mvinCtpvNm":"from_province",
    "mvtCtpvNm":"to_province",
    "mvinSggNm":"from_district",
    "mvtSggNm":"to_district",
    "totNmprCnt":"total_people",
    "maleNmprCnt":"male",
    "femlNmprCnt":"female"})


In [ ]:
def get_population_flow(districts, start_date, end_date):
    dfs = []
    completed = 0
    total = len(districts) ** 2
    max_retries = 3

    for origin in districts:
        for destination in districts:

            param = {
                "serviceKey": POP_FLOW_SERVICE_KEY,
                "mvinAdmmCd": districts[origin],
                "mvtAdmmCd": districts[destination],
                "srchFrYm": start_date,
                "srchToYm": end_date,
                "lv": 2,
                "type": "XML",
                "numOfRows": BATCH_MONTHS,
                "pageNo": 1
            }

            for attempt in range(max_retries):
                try:
                    response = requests.get(
                        POP_FLOW_BASE_URL,
                        params=param,
                        timeout=30
                    )

                    response.raise_for_status()

                    root = ET.fromstring(response.text)

                    df = rename_columns(return_df(root))
                    dfs.append(df)

                    completed += 1

                    print(
                        f"{completed}/{total}: "
                        f"Fetched {origin} to {destination}"
                    )

                    break

                except ReadTimeout:
                    print(
                        f"Timeout: {origin} -> {destination} "
                        f"(attempt {attempt + 1}/{max_retries})"
                    )

                    if attempt < max_retries - 1:
                        time.sleep(2)

            else:
                raise RuntimeError(
                    f"Failed after {max_retries} attempts: "
                    f"{origin} -> {destination}"
                )
    print(f"Extraction complete: {completed} rows")
    return pd.concat(dfs, ignore_index=True)
    
def generate_date_batches():
    current = pd.to_datetime(START_DATE, format='%Y%m')
    end = pd.to_datetime(END_DATE, format='%Y%m')

    date_batches = []

    while current <= end:
        batch_end = min(
            current + pd.DateOffset(months=2),
            end
        )

        current_batch = [
            current.strftime('%Y%m'),
            batch_end.strftime('%Y%m')
            ]

        date_batches.append(current_batch)   

        current = batch_end + pd.DateOffset(months=1)
    return date_batches



In [4]:
create_seoul_population_flow_table_sql = '''
CREATE TABLE IF NOT EXISTS seoul_population_flow (
    date INT,
    from_province VARCHAR(50),
    to_province VARCHAR(50),
    from_district VARCHAR(50),
    to_district VARCHAR(50),
    total_people INT,
    male INT,
    female INT,
    PRIMARY KEY (date, from_province, to_province, from_district, to_district)
)
'''

insert_seoul_population_flow_sql = '''
INSERT INTO seoul_population_flow (
    date,
    from_province,
    to_province,
    from_district,
    to_district,
    total_people,
    male,
    female
) VALUES (
    :date,
    :from_province,
    :to_province,
    :from_district,
    :to_district,
    :total_people,
    :male,
    :female
)
ON CONFLICT (date, from_province, to_province, from_district, to_district) DO NOTHING
'''



In [8]:
def load_seoul_population_flow(conn, df):
    records = df.to_dict(orient='records')
    batch_size = 100
    for i in range(0,len(records),batch_size):
      batch = records[i:i + batch_size]
      conn.execute(text(insert_seoul_population_flow_sql),batch)

      print(f"{min(i + batch_size,len(records))}/{len(records)} inserted")



def main():
   date_batches = generate_date_batches()
   districts = read_districts_csv()
   engine = create_db_engine()

   for start,end in date_batches[1:2]:
      print(f"Starting batch: {start} -> {end}")

      result = get_population_flow(districts,start,end)
      for attempt in range(3):
         try: 
            with engine.begin() as conn:
               load_seoul_population_flow(conn,result)

            break
         except OperationalError:
            print(f"DB connection failed. Retry {attempt+1}/3")

      else:
         raise RuntimeError(
            f"Failed to load batch {start} -> {end} after 3 attempts"
         )

      print(f"Completed batch: {start} -> {end}")

main()

Starting batch: 202304 -> 202306
1/625: Successfully loaded 중구 to 중구
2/625: Successfully loaded 중구 to 종로구
3/625: Successfully loaded 중구 to 용산구
4/625: Successfully loaded 중구 to 성동구
5/625: Successfully loaded 중구 to 광진구
6/625: Successfully loaded 중구 to 동대문구
7/625: Successfully loaded 중구 to 중랑구
8/625: Successfully loaded 중구 to 성북구
9/625: Successfully loaded 중구 to 강북구
10/625: Successfully loaded 중구 to 도봉구
11/625: Successfully loaded 중구 to 노원구
12/625: Successfully loaded 중구 to 은평구
13/625: Successfully loaded 중구 to 서대문구
14/625: Successfully loaded 중구 to 마포구
15/625: Successfully loaded 중구 to 양천구
16/625: Successfully loaded 중구 to 강서구
17/625: Successfully loaded 중구 to 구로구
18/625: Successfully loaded 중구 to 금천구
19/625: Successfully loaded 중구 to 영등포구
20/625: Successfully loaded 중구 to 동작구
21/625: Successfully loaded 중구 to 관악구
22/625: Successfully loaded 중구 to 서초구
23/625: Successfully loaded 중구 to 강남구
24/625: Successfully loaded 중구 to 송파구
25/625: Successfully loaded 중구 to 강동구
26/625: Successfully loa